# BERTopic 动态主题追踪实验

本实验使用 BERTopic 对知乎 AI 圈子数据进行动态主题建模，追踪热点话题随时间的演化。

## 技术栈
- **BERTopic**: 基于 BERT + c-TF-IDF 的主题建模
- **Sentence Transformers**: 文本嵌入
- **UMAP**: 降维
- **HDBSCAN**: 密度聚类

In [25]:
# 安装必要的库 (首次运行时取消注释)
# !pip install bertopic -i https://pypi.tuna.tsinghua.edu.cn/simple
# !pip install sentence-transformers -i https://pypi.tuna.tsinghua.edu.cn/simple
# !pip install umap-learn -i https://pypi.tuna.tsinghua.edu.cn/simple
# !pip install hdbscan -i https://pypi.tuna.tsinghua.edu.cn/simple
# !pip install jieba -i https://pypi.tuna.tsinghua.edu.cn/simple
# !pip install nbformat -i https://pypi.tuna.tsinghua.edu.cn/simple  # 用于显示 plotly 图表

In [26]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# tqdm 进度条
from tqdm.auto import tqdm
tqdm.pandas()  # 启用 pandas 支持

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. 数据加载与预处理

In [27]:
# 加载数据
data_dir = Path("../data")
data_file = data_dir / "zhihu_ring_data_20260225_senti.json"
circles_file = data_dir / "zhihu_ai_circles.json"

with open(data_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

with open(circles_file, 'r', encoding='utf-8') as f:
    circles_data = json.load(f)

# 创建圈子ID到名称的映射
ring_to_name = {c['ring_id']: c['name'] for c in circles_data}

# 转换为 DataFrame
df = pd.DataFrame(data)
df['ring_name'] = df['ring_id'].map(ring_to_name).fillna(df['ring_id'])

# 转换时间戳
df['pub_time'] = pd.to_datetime(df['pub_time'])
df['date'] = df['pub_time'].dt.date
df['year_month'] = df['pub_time'].dt.to_period('M')

# 基本统计
print(f"总数据量: {len(df)} 条")
print(f"时间范围: {df['pub_time'].min()} 至 {df['pub_time'].max()}")
print(f"\n圈子分布 (Top 10):")
print(df['ring_name'].value_counts().head(10))

总数据量: 3009 条
时间范围: 2025-03-21 18:43:00 至 2026-02-17 13:07:00

圈子分布 (Top 10):
ring_name
AI与人类未来          466
AI 创投生态圈         263
科研 AI Hub        255
算法研究所            253
AI写作研究所          246
AI Coding 探索舰    241
AI 工具测评中心        237
AI 时代的我们         226
AI 安全研究所         214
DeepSeek 深潜舱     168
Name: count, dtype: int64


In [28]:
# 文本预处理
import re
import jieba

def clean_text(text):
    """清洗文本"""
    # 去除HTML标签
    text = re.sub(r'<[^>]+>', '', text)
    # 去除URL
    text = re.sub(r'https?://\\S+', '', text)
    # 去除多余空白
    text = re.sub(r'\\s+', ' ', text)
    # 去除话题标签
    text = re.sub(r'#[^#\\s]+#?', '', text)
    return text.strip()

def chinese_tokenize(text):
    """中文分词函数"""
    words = jieba.lcut(text)
    # 过滤停用词和单字
    stop_words = {'的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去',
                  '你', '会', '着', '没有', '看', '好', '自己', '这', '那', '与', '对于', '为了', '因为', '所以', '但是', '虽然',
                  '可以', '这个', '那个', '什么', '怎么', '如何', '阅读', '全文', '链接', '分享', '觉得', '感觉', '其实',
                  '如果', '只要', '然后', '最后', '比如', '像', '还是', '或者', '而且', '不过', '当然', '可能', ' ', ''}
    return [w for w in words if w not in stop_words and len(w) > 1]

# 应用清洗 (带进度条)
print("清洗文本...")
df['content_clean'] = df['content'].progress_apply(clean_text)

# 分词 (用于 CountVectorizer)
print("分词中...")
df['content_tokenized'] = df['content_clean'].progress_apply(
    lambda x: ' '.join(chinese_tokenize(x))
)

# 准备建模数据
docs = df['content_clean'].tolist()  # 用于嵌入生成 (原始文本)
docs_tokenized = df['content_tokenized'].tolist()  # 用于主题建模 (已分词)
timestamps = df['pub_time'].tolist()

print(f"文档数量: {len(docs)}")
print(f"平均文本长度: {np.mean([len(d) for d in docs]):.0f} 字符")
print(f"示例分词结果: {docs_tokenized[0][:100]}...")

清洗文本...


100%|██████████| 3009/3009 [00:00<00:00, 239258.77it/s]


分词中...


100%|██████████| 3009/3009 [00:02<00:00, 1463.92it/s]

文档数量: 3009
平均文本长度: 281 字符
示例分词结果: AI 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 AI 需求 减少 AI 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年 打工 ...


## 2. BERTopic 模型配置

### 2.1 模型选择

本地可用的嵌入模型：

| 模型 | 类型 | 路径 | 推荐用途 |
|------|------|------|----------|
| **text2vec-base-chinese** | Sentence Transformer | `src/models/text2vec-base-chinese` | ✅ 推荐，专为句子嵌入设计 |
| **chinese-roberta-wwm-ext** | RoBERTa | `src/models/chinese-roberta-wwm-ext` | 需额外处理，不如 text2vec 方便 |

**为什么推荐 text2vec？**
- 已经过 mean pooling 训练，直接输出句子级嵌入
- RoBERTa 输出的是 token 级嵌入，需要额外处理才能用于句子相似度

In [29]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import os

# 获取项目根目录
PROJECT_ROOT = Path(os.path.abspath('.')).parent

# 使用本地中文 Sentence Transformer 模型
model_path = PROJECT_ROOT / 'src' / 'models' / 'text2vec-base-chinese'
print(f"加载本地模型: {model_path}")

# 检查本地模型是否存在
if model_path.exists():
    embedding_model = SentenceTransformer(str(model_path))
    print("本地模型加载成功!")
else:
    # 如果本地不存在，从 HuggingFace 下载
    print("本地模型不存在，从 HuggingFace 下载...")
    embedding_model = SentenceTransformer('shibing624/text2vec-base-chinese')
    print("模型下载完成!")

# 配置 CountVectorizer - 重要：针对已分词的中文
# 因为会在分词步骤中将词用空格分隔，所以这里使用宽松的 pattern
vectorizer_model = CountVectorizer(
    max_features=1000,
    token_pattern=r'(?u).+'  # 匹配任何非空字符串（适合已分好词的中文）
)

# 注意：中文文本需要分词后才能用于 c-TF-IDF
# - docs: 原始文本，用于生成 embedding
# - docs_tokenized: jieba 分词后的文本，用于 BERTopic 的主题表示 (c-TF-IDF)

加载本地模型: d:\Projects\nlp-public-opinion-analysis\src\models\text2vec-base-chinese
本地模型加载成功!


### 2.2 初始化 BERTopic

In [30]:
# UMAP 参数
umap_model = UMAP(
    n_neighbors=25,
    n_components=50,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

# HDBSCAN 参数
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=15,
    metric='euclidean',
    prediction_data=True
)

# 创建 BERTopic 模型
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True,
    nr_topics='auto'  # 自动减少主题数量
)

print("BERTopic model initialized!")

BERTopic model initialized!


## 3. 生成嵌入与训练模型

In [31]:
# 生成文档嵌入
print("Generating embeddings...")
embeddings = embedding_model.encode(
    docs, 
    show_progress_bar=True, 
    batch_size=64
)

print(f"Embeddings shape: {embeddings.shape}")
print("嵌入生成完成!")

Generating embeddings...


Batches: 100%|██████████| 48/48 [02:10<00:00,  2.72s/it]

Embeddings shape: (3009, 768)
嵌入生成完成!


In [32]:
# 训练模型
print("Training BERTopic model...")
print("-" * 50)

# 重要：使用 docs_tokenized 而不是 docs
# BERTopic 用传入的文档做 c-TF-IDF 主题表示，需要用分词后的中文
topics, probs = topic_model.fit_transform(docs_tokenized, embeddings=embeddings)

print("-" * 50)
print(f"\n发现 {len(set(topics)) - (1 if -1 in topics else 0)} 个主题")
print(f"噪声文档数: {list(topics).count(-1)} ({list(topics).count(-1)/len(topics)*100:.1f}%)")

2026-03-04 21:24:00,692 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model...
--------------------------------------------------


2026-03-04 21:24:31,743 - BERTopic - Dimensionality - Completed ✓
2026-03-04 21:24:31,743 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-04 21:24:32,314 - BERTopic - Cluster - Completed ✓
2026-03-04 21:24:32,315 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-04 21:24:32,338 - BERTopic - Representation - Completed ✓
2026-03-04 21:24:32,339 - BERTopic - Topic reduction - Reducing number of topics
2026-03-04 21:24:32,346 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-04 21:24:32,363 - BERTopic - Representation - Completed ✓
2026-03-04 21:24:32,365 - BERTopic - Topic reduction - Reduced number of topics from 3 to 3


--------------------------------------------------

发现 2 个主题
噪声文档数: 25 (0.8%)


In [33]:
# 查看主题信息
topic_info = topic_model.get_topic_info()
print("\n=== 主题概览 ===")
print(topic_info.head(15))


=== 主题概览 ===
   Topic  Count                                                                                                 Name  \
0     -1     25  -1_张小北 关雅荻 深度 探讨 ai 影像 电影 行业 冲击 最新 ai 影像 放映室 ai 行业 产生 巨大 冲击 直接 反应 春节前夕 关雅荻 进行 一场 关于 ai 影像 技术 电影 ...   
1      0   2951  0_ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 ...   
2      1     33  1_散列 哈希 技术 bingo 起来 知乎 圈子 参与 bingo 挑战 圈内人 哈希 散列 技术 bingo 开始 游戏 快来测 圈层 浓度 区块 游戏 一起 知乎 圈子 参与 bingo...   

                                                                                        Representation  \
0  [张小北 关雅荻 深度 探讨 ai 影像 电影 行业 冲击 最新 ai 影像 放映室 ai 行业 产生 巨大 冲击 直接 反应 春节前夕 关雅荻 进行 一场 关于 ai 影像 技术 电影 行业...   
1  [ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十...   
2  [散列 哈希 技术 bingo 起来 知乎 圈子 参与 bingo 挑战 圈内人 哈希 散列 技术 bingo 开始 游戏 快来测 圈层 浓度 区块 游戏 一起 知乎 圈子 参与 bingo ...   

                                                                                

In [34]:
# 查看每个主题的关键词
print("\n=== 主题关键词 ===")
for topic_id in topic_info['Topic'].values:
    if topic_id == -1:
        continue
    if topic_id > 15:  # 只显示前15个主题
        break
    topic_words = topic_model.get_topic(topic_id)
    words = ', '.join([word for word, score in topic_words[:5]])
    count = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
    print(f"Topic {topic_id:2d} ({count:4d} docs): {words}")


=== 主题关键词 ===
Topic  0 (2951 docs): ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年 打工 打工 十年 科技 小镇 智能化 生产 工厂 雇佣 少量 人员 创造 大量 财富 产业链 几万人 团队 能够 提供 几百万 千万 物质 生活 财富 因此 人类 社会 面临 劳动力 过剩 产能 过剩 窘境 无条件 基本 收入 保障 成为 未来 社会 发展 方向 中国式 ubi 三年 兴趣 基本 收入 保障 旨在 培养 公民 兴趣爱好 创造力 提升 未来 国际 竞争力 ai 导致 大规模 失业 说法 是不是 悖论 ai 时代 普通人 应该 何处 何从 人工智能 时代 来临 ai 取代 人类 中国式 ubi ai 时代 最该 担心 不是 失业 最近 大火 seedance 2.0 真的 超出 想象 直接 碾压 很多 专业 影视制作 视频 门槛 几乎 ai 发展 越来越快 很多 一技之长 开始 大家 焦虑 不是 失业 不是 不到 而是 吃饭 技能 没用 实际上 不光 简单劳动 替代 高级 组合 技能 替代 这才 最让人 害怕 地方 在我看来 人类 一样 东西 ai 很难 替代 就是 真实 经历 原始 想法 ai 没法 真正 活过 人生 正面 这是 我们 底气 反面 我们 变得 模板 一点 优势 消失 普通人 到底 应对 我分 三点 扔掉 观念 ai 人工 智障 现在 ai 已经 很强 承认 能力 开始 就会输 立刻 ai 插进 工作 不用 专门 去学 ai 直接 改造 现在 工作 想法 ai 实现 写作 ai 视频 创意 ai 降低成本 这才 真正 拥抱 ai 千万别 很多 人用 沉迷 研究 工具 核心 经历 思想 在我看来 这是 最蠢 做法 ai 助手 不是 主人 一句 现实 的话 ai 眼里 就是 一堆 数据 数据 提供者 数据 标注 指挥者 决策者 真实 经验 创作者 技能 替代 经历 思想 永远 独一无二 ai kimi openclaw 搬进 浏览器 暗面 昨天 正式 发布 kimi claw beta 简单 来说 就是 openclaw 直接 内置 kimi 浏览器 ai agent 不用 服务

## 4. 动态主题追踪

In [35]:
# 计算主题随时间的变化
nr_bins = min(20, len(df['year_month'].unique()))  # 时间窗口数量

print(f"计算主题随时间变化 (时间窗口数: {nr_bins})...")

# 使用 docs_tokenized 而不是 docs
topics_over_time = topic_model.topics_over_time(
    docs_tokenized,
    timestamps,
    nr_bins=nr_bins,
    global_tuning=True,
    evolution_tuning=True
)

print("\nTopics over time computed!")

计算主题随时间变化 (时间窗口数: 12)...


12it [00:00, 82.71it/s]


Topics over time computed!


In [36]:
# 查看主题随时间变化的数据
print("\n=== 主题随时间变化数据 ===")
print(topics_over_time.head(20))


=== 主题随时间变化数据 ===
    Topic                                                                                                Words  Frequency               Timestamp
0       0  ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年...          1 2025-03-21 10:43:48.960
1       0  ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年...          8 2025-04-18 12:15:00.000
2       0  ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年...         70 2025-05-16 05:47:00.000
3       0  ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年...         80 2025-06-12 23:19:00.000
4       0  ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 十年...        115 2025-07-10 16:51:00.000
5       0  ai 导致 大规模 失业 说法 是不是 悖论 人类 大量 失业 不会 导致 ai 需求 减少 ai 生产 服务 仍能 满足 普通人 消费 需求 未来 智能 机器人 时代 工作 方式 变为 

## 5. 可视化

In [37]:
# 创建输出目录
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

### 5.1 主题时间演化图

In [38]:
# 可视化主题随时间的变化
fig_time = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=12,
    width=1200,
    height=600
)

# 保存为 HTML (始终执行)
fig_time.write_html(str(output_dir / "bertopic_topics_over_time.html"))
print(f"图表已保存至: {output_dir / 'bertopic_topics_over_time.html'}")

# 尝试在 notebook 中显示 (需要 nbformat)
try:
    fig_time.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("提示: 运行 `pip install nbformat` 后可在 notebook 中直接显示交互式图表")
    else:
        raise

图表已保存至: ..\outputs\bertopic_topics_over_time.html
提示: 运行 `pip install nbformat` 后可在 notebook 中直接显示交互式图表


### 5.2 主题分布图

In [39]:
# 可视化主题分布
fig_topics = topic_model.visualize_topics(
    width=1000,
    height=700
)

# 保存
fig_topics.write_html(str(output_dir / "bertopic_topics_distribution.html"))

# 尝试显示
try:
    fig_topics.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("主题分布图已保存! (运行 `pip install nbformat` 可在 notebook 中显示)")
    else:
        raise
else:
    print("主题分布图已保存!")

ValueError: zero-size array to reduction operation maximum which has no identity

### 5.3 主题相似度热力图

In [ ]:
# 可视化主题相似度热力图
fig_heatmap = topic_model.visualize_heatmap(
    n_clusters=10,
    top_n_topics=20,
    width=1000,
    height=800
)

# 保存
fig_heatmap.write_html(str(output_dir / "bertopic_heatmap.html"))

# 尝试显示
try:
    fig_heatmap.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("相似度热力图已保存! (运行 `pip install nbformat` 可在 notebook 中显示)")
    else:
        raise
else:
    print("相似度热力图已保存!")

## 6. 按圈子分析主题分布

In [ ]:
# 计算每个圈子的主题分布
print("计算各圈子的主题分布...")

# 使用 docs_tokenized 而不是 docs
topics_per_class = topic_model.topics_per_class(
    docs_tokenized,
    classes=df['ring_name'].tolist(),
    global_tuning=True
)

# 可视化
fig_class = topic_model.visualize_topics_per_class(
    topics_per_class,
    top_n_topics=10,
    width=1200,
    height=700
)

# 保存
fig_class.write_html(str(output_dir / "bertopic_topics_per_circle.html"))

# 尝试显示
try:
    fig_class.show()
except ValueError as e:
    if "nbformat" in str(e):
        print("按圈子主题分布已保存! (运行 `pip install nbformat` 可在 notebook 中显示)")
    else:
        raise
else:
    print("按圈子主题分布已保存!")

## 7. 主题强度演化分析

In [ ]:
# 分析每个主题在不同时间段的强度变化
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 获取主要主题 (排除噪声 -1)
major_topics = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)['Topic'].head(10).tolist()

# 过滤数据
tot_data = topics_over_time[topics_over_time['Topic'].isin(major_topics)]

# 创建折线图
fig, ax = plt.subplots(figsize=(14, 8))

for topic in major_topics:
    topic_data = tot_data[tot_data['Topic'] == topic].sort_values('Timestamp')
    if len(topic_data) > 0:
        # 获取主题关键词
        words = topic_model.get_topic(topic)
        label = ', '.join([w for w, s in words[:3]])
        ax.plot(topic_data['Timestamp'], topic_data['Frequency'], marker='o', label=f"Topic {topic}: {label}", linewidth=2)

ax.set_xlabel('时间', fontsize=12)
ax.set_ylabel('话题强度 (Frequency)', fontsize=12)
ax.set_title('主要话题强度随时间演化', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

# 保存图片
plt.savefig(output_dir / "topic_evolution_line.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"主题演化折线图已保存至: {output_dir / 'topic_evolution_line.png'}")

## 8. 查看代表性文档

In [ ]:
def get_representative_docs(topic_model, docs, topic_id, top_k=3):
    """获取特定主题的代表性文档
    
    注意：这里用 docs (原始文本) 显示内容，更易读
    但主题建模用的是 docs_tokenized (分词后)
    """
    # 获取该主题的所有文档索引
    topic_indices = [i for i, t in enumerate(topics) if t == topic_id]
    
    if len(topic_indices) == 0:
        return []
    
    # 获取主题词
    topic_words = topic_model.get_topic(topic_id)
    words_str = ', '.join([w for w, s in topic_words[:5]])
    
    print(f"\n=== Topic {topic_id} 关键词: {words_str} ===")
    print(f"共 {len(topic_indices)} 篇文档\n")
    
    # 返回前 top_k 篇文档 (使用原始文本显示)
    for idx in topic_indices[:top_k]:
        print(f"--- 文档 {idx} ---")
        print(f"圈子: {df.loc[idx, 'ring_name']}")
        print(f"时间: {df.loc[idx, 'pub_time']}")
        print(f"内容: {docs[idx][:200]}...\n")

# 查看前5个主题的代表性文档
major_topics_for_reps = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)['Topic'].head(5).tolist()

print("获取代表性文档...")
for topic_id in tqdm(major_topics_for_reps, desc="Extracting Representative Docs"):
    get_representative_docs(topic_model, docs, topic_id, top_k=2)

## 9. 模型保存

In [ ]:
# 保存模型
model_path = output_dir / "bertopic_model"
topic_model.save(str(model_path))
print(f"模型已保存至: {model_path}")

# 加载模型 (示例)
# loaded_model = BERTopic.load(str(model_path))
# print("模型加载成功!")

## 10. 参数调优 (可选)

In [ ]:
# 测试不同的参数组合
import itertools
from tqdm.auto import tqdm

results = []

# 参数组合
param_combinations = list(itertools.product([20, 30, 50], [15, 25]))

print(f"参数调优实验: 共 {len(param_combinations)} 组参数\n")

for min_cluster_size, n_neighbors in tqdm(param_combinations, desc="Parameter Tuning"):
    print(f"Testing: min_cluster_size={min_cluster_size}, n_neighbors={n_neighbors}")
    
    # 创建模型
    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=50,
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )
    
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_cluster_size // 2,
        metric='euclidean',
        prediction_data=True
    )
    
    model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        verbose=False
    )
    
    # 使用 docs_tokenized
    topics_test, _ = model.fit_transform(docs_tokenized, embeddings=embeddings)
    
    n_clusters = len(set(topics_test)) - (1 if -1 in topics_test else 0)
    n_noise = list(topics_test).count(-1)
    
    results.append({
        'min_cluster_size': min_cluster_size,
        'n_neighbors': n_neighbors,
        'n_topics': n_clusters,
        'noise_count': n_noise,
        'noise_ratio': f"{n_noise/len(topics_test)*100:.1f}%"
    })

# 显示结果
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("参数对比结果")
print("="*60)
print(results_df.to_string(index=False))

## 11. 总结

本实验使用 BERTopic 完成了以下任务：

1. **主题发现**: 使用 BERT + c-TF-IDF 发现知乎 AI 数据中的主要话题
2. **动态追踪**: 使用 `topics_over_time()` 方法追踪话题随时间的演化
3. **可视化**: 生成话题分布图、相似度热力图、时间演化图等
4. **按圈子分析**: 分析不同圈子的主题偏好

### 与 SentenceBERT + UMAP + HDBSCAN 的对比

| 特性 | 之前的方法 | BERTopic |
|------|-----------|----------|
| 主题表示 | 普通 TF-IDF | c-TF-IDF (类级别) |
| 动态追踪 | 需手动实现 | 内置 `topics_over_time()` |
| 可视化 | 需自定义 | 12 种内置可视化 |
| 易用性 | 需自己组合 | 一站式 API |

### 下一步改进方向

1. **增量学习**: 对新数据进行增量更新，无需重新训练
2. **主题合并**: 手动合并相似的主题
3. **自定义表示**: 使用 KeyBERT 或 LLM 优化主题描述
4. **多模态**: 结合图像进行多模态主题建模